# Auditoria do de-para ICTV/NCBI versus split de fragmentos

Lê **o Parquet real** `main_taxonomy_hybrid_CLEANSED.parquet` produzido por `short/ICTV_improve_kraken_database.ipynb`, confere sua identidade de rótulos com `dataset_kraken/species_name_ids.csv` e cruza as classes com `per_species.csv` produzido pela auditoria de treino/teste. Estratifica duplicatas exatas por espécie, gênero, família e ordem.

A planilha ICTV e o fallback do NCBI aqui descrevem **táxons** (uma linha por espécie), não accessions de genomas de origem dos 15 milhões de fragmentos. Portanto esta auditoria identifica problemas do de-para e distribuição taxonômica da sobreposição, mas **não certifica independência por genoma e não toma a decisão final de retreinamento**. Não usa dados simulados nem chama a API do NCBI.


In [10]:
import os
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
search_roots = ([Path(os.environ["CNN_METAGENOME_ROOT"]).expanduser().resolve()]
                if os.getenv("CNN_METAGENOME_ROOT") else [])
search_roots += [current_dir, current_dir / "cnn_metagenome", *current_dir.parents]
for candidate in search_roots:
    if (candidate / "src" / "viral_cnn").is_dir():
        PROJECT_ROOT = candidate
        sys.path.insert(0, str(candidate / "src"))
        break
else:
    raise FileNotFoundError("Abra este notebook a partir do repositório cnn_metagenome")

from viral_cnn.data.taxonomy_split_audit import audit_taxonomy_against_overlap

if os.getenv("SMOKE_TEST", "0").lower() in ("1", "true", "yes"):
    raise RuntimeError("SMOKE_TEST=1 impede uma auditoria científica; use os arquivos reais")


## 1. Identifique os artefatos gerados

O notebook original salva `main_taxonomy_hybrid_CLEANSED.parquet` no seu diretório de execução (ou `cnn_kraken/data_taxonomy_parquet/`. A busca cobre ambos e aceita `main_taxonomy_hybrid_ictv_ncbi.parquet` quando não houver uma versão `CLEANSED`. Você também pode definir `TAXONOMY_PATH` como caminho do arquivo ou do diretório. `OVERLAP_DIR` aponta para a saída da auditoria anterior. O CSV de rótulos da base original deve ser o mesmo que alimentou o notebook ICTV.


In [11]:
# Célula autônoma: funciona após reiniciar o kernel ou executá-la isoladamente.
import os
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
search_roots = ([Path(os.environ["CNN_METAGENOME_ROOT"]).expanduser().resolve()]
                if os.getenv("CNN_METAGENOME_ROOT") else [])
search_roots += [current_dir, current_dir / "cnn_metagenome", *current_dir.parents]
for candidate in search_roots:
    if (candidate / "src" / "viral_cnn").is_dir():
        PROJECT_ROOT = candidate
        if str(candidate / "src") not in sys.path:
            sys.path.insert(0, str(candidate / "src"))
        break
else:
    raise FileNotFoundError("Execute o notebook dentro de cnn_metagenome (ou informe a raiz via diretório de trabalho)")

from viral_cnn.data.taxonomy_split_audit import audit_taxonomy_against_overlap
if os.getenv("SMOKE_TEST", "0").lower() in ("1", "true", "yes"):
    raise RuntimeError("SMOKE_TEST=1 impede uma auditoria científica; use os arquivos reais")

# O notebook ICTV pode salvar antes e depois do saneamento em locais distintos.
TAXONOMY_NAMES = ("main_taxonomy_hybrid_CLEANSED.parquet",
                  "main_taxonomy_hybrid_ictv_ncbi.parquet")
DATA_ROOT = Path(os.getenv("DATA_ROOT", str(PROJECT_ROOT.parent))).expanduser().resolve()
taxonomy_dir = DATA_ROOT / "cnn_kraken" / "data_taxonomy_parquet"
search_dirs = [taxonomy_dir, PROJECT_ROOT / "short", PROJECT_ROOT]
configured = os.getenv("TAXONOMY_PATH")
if configured:
    selected = Path(configured).expanduser().resolve()
    candidates = ([selected / name for name in TAXONOMY_NAMES]
                  if selected.is_dir() else [selected])
else:
    # Prioriza a versão saneada em qualquer diretório; usa o Parquet híbrido se for a única versão.
    candidates = [directory / name for name in TAXONOMY_NAMES for directory in search_dirs]
TAXONOMY_PATH = next((file for file in candidates if file.is_file()), None)
if TAXONOMY_PATH is None:
    available = sorted(str(file) for file in taxonomy_dir.glob("*.parquet")) if taxonomy_dir.is_dir() else []
    raise FileNotFoundError(
        f"De-para taxonômico não encontrado. Candidatos: {[str(file) for file in candidates]}. "
        f"Parquets presentes em {taxonomy_dir}: {available}. "
        "Defina TAXONOMY_PATH com o diretório ou o arquivo correto."
    )

ORIGINAL_MAPPING = Path(os.getenv("ORIGINAL_MAPPING_PATH", str(PROJECT_ROOT / "dataset_kraken" / "species_name_ids.csv"))).expanduser()
OVERLAP_DIR = Path(os.getenv("OVERLAP_DIR", str(PROJECT_ROOT / "results_reproduction" / "retraining_decision"))).expanduser()
OUTPUT_DIR = Path(os.getenv("TAXONOMY_AUDIT_OUTPUT_DIR", str(PROJECT_ROOT / "results_reproduction" / "ictv_split_audit"))).expanduser()
for required in (TAXONOMY_PATH, ORIGINAL_MAPPING, OVERLAP_DIR / "decision.json", OVERLAP_DIR / "per_species.csv"):
    if not required.is_file():
        raise FileNotFoundError(f"Arquivo real não encontrado: {required}. Ajuste TAXONOMY_PATH / OVERLAP_DIR / ORIGINAL_MAPPING_PATH")
print("Parquet ICTV:", TAXONOMY_PATH)
print("De-para original:", ORIGINAL_MAPPING)
print("Resultado da auditoria de overlap:", OVERLAP_DIR)


Parquet ICTV: /home/mlima/workspace/cnn_kraken/data_taxonomy_parquet/main_taxonomy_hybrid_ictv_ncbi.parquet
De-para original: /home/mlima/workspace/cnn_metagenome/dataset_kraken/species_name_ids.csv
Resultado da auditoria de overlap: /home/mlima/workspace/cnn_metagenome/results_reproduction/retraining_decision


## 2. Faça o cruzamento e leia os limites

O relatório reprova IDs ausentes ou duplicados e compara `species` por `id` com o mapeamento anterior à curadoria. Mudanças em `species_validated_name` são permitidas: são justamente o de-para. As taxas por gênero/família/ordem são ponderadas pelos fragmentos observados em cada classe. A auditoria antiga, calculada antes de remover padding `_`, é marcada explicitamente como **limite inferior**, não taxa final.


In [12]:
report = audit_taxonomy_against_overlap(
    taxonomy_path=TAXONOMY_PATH,
    per_species_path=OVERLAP_DIR / "per_species.csv",
    decision_path=OVERLAP_DIR / "decision.json",
    output_dir=OUTPUT_DIR,
    original_mapping_path=ORIGINAL_MAPPING,
)
print("Status:", report["status"])
print("Integridade do de-para:", report["mapping_integrity_ok"])
print("Classes sem de-para:", len(report["missing_test_labels"]))
print("IDs duplicados:", len(report["duplicate_taxonomy_labels"]))
print("Rótulos originais com nome alterado:", len(report["original_label_name_mismatch"]))
print("Correspondência exata/RC:", report["exact_overlap_pct"], "%",
      f'({report["overlap_rate_interpretation"]})')
print("Genoma de origem determinado pelo de-para:", report["genome_provenance_resolved"])
print("Origem da taxonomia:", report["taxonomy_source_counts"])
print("Saídas:", OUTPUT_DIR / "taxonomy_audit.json", OUTPUT_DIR / "per_taxon.csv",
      OUTPUT_DIR / "per_genus.csv", OUTPUT_DIR / "per_family.csv", OUTPUT_DIR / "per_order.csv")


Status: TAXONOMY_ONLY_PROVENANCE_REQUIRED
Integridade do de-para: True
Classes sem de-para: 0
IDs duplicados: 0
Rótulos originais com nome alterado: 0
Correspondência exata/RC: 95.9362 % (corrected_full_test)
Genoma de origem determinado pelo de-para: False
Origem da taxonomia: {'ICTV Master': 7746, 'NCBI Fallback': 5937}
Saídas: /home/mlima/workspace/cnn_metagenome/results_reproduction/ictv_split_audit/taxonomy_audit.json /home/mlima/workspace/cnn_metagenome/results_reproduction/ictv_split_audit/per_taxon.csv /home/mlima/workspace/cnn_metagenome/results_reproduction/ictv_split_audit/per_genus.csv /home/mlima/workspace/cnn_metagenome/results_reproduction/ictv_split_audit/per_family.csv /home/mlima/workspace/cnn_metagenome/results_reproduction/ictv_split_audit/per_order.csv


## 3. Como usar a conclusão

- **Se `mapping_integrity_ok` for falso:** corrigir IDs, nomes originais ou classes ausentes antes de reutilizar métricas por hierarquia.
- **Se o mapeamento estiver íntegro:** o mesmo label continua correspondendo ao mesmo táxon, e os CSVs permitem quantificar em quais clados ocorrem correspondências exatas. Isso não identifica a referência genômica que produziu cada fragmento.
- **Para generalização a genomas não vistos:** localizar a FASTA de origem com headers/accessions, restaurar o vínculo `fragmento → accession` com correspondências inequívocas e então criar split por accession. Registros ambíguos devem permanecer não atribuídos.
- **Se `overlap_rate_interpretation` for `legacy_raw_hash_lower_bound_rerun_recommended`:** reexecute o notebook 10 corrigido para comparar sequências após remover o padding; depois rode esta auditoria novamente.
